In [3]:
#getting the precision matrix

import numpy as np
import pandas as pd

matrix = pd.read_csv("matrix_of_sig_gene_gene_correlations.tsv", sep='\t', index_col=0) #index_col = 0 means the first column is an index
matrix = matrix.astype(float)   #makes sure all the data types are floats 

# Add small value to diagonal for stability
regularized = matrix.values.copy()    #turns the df into a numpy array which is then deep copied 
np.fill_diagonal(regularized, regularized.diagonal() + 0.01)   #adds 0.01 to each diagonal entry to stabalize matrix 
"""
When you invert a matrix, the math requires dividing by certain values derived from the matrix. If your matrix has variables that are very highly correlated, some of these values get extremely close to zero, which causes division by near-zero — making the result numerically unstable or completely wrong.
Think of it like dividing by 0.000001 vs dividing by 1 — tiny differences in the input cause huge swings in the output.
Adding 0.01 to the diagonal nudges these near-zero values away from zero, so the division becomes stable. The diagonal represents each variable's variance with itself, so you're essentially saying "pretend each variable has just a tiny bit more independent variation than it actually does."
The tradeoff is a very slight distortion of the results, but 0.01 is small enough that the partial correlations you get out are still accurate — you're just trading a tiny bit of precision for a lot of numerical stability.
"""

# Then invert
precision_matrix = np.linalg.inv(regularized)

# Convert precision matrix back to dataframe with gene names as labels
precision_df = pd.DataFrame(precision_matrix, index=matrix.index, columns=matrix.columns)
precision_df.head()

# Save to tsv
precision_df.to_csv('precision_matrix.tsv', sep='\t', index=True)

In [4]:
#getting the partial correlation matrix from the precision matrix

# Get diagonal values as a 1d np array
diag = np.diag(precision_matrix)

# Apply normalization formula: -P_ij / sqrt(P_ii * P_jj)
partial_corr = np.zeros_like(precision_matrix)    #creates a matrix of zeros with the same shape(num rows and num columns) as precision matrix
for i in range(len(precision_matrix)):
    for j in range(len(precision_matrix)):
        if i != j:  # skip diagonal
            partial_corr[i, j] = -precision_matrix[i, j] / np.sqrt(diag[i] * diag[j])  #storing the calculation of the partial correlation in the correct slot of the matrix.

# Convert to dataframe with gene names
partial_corr_df = pd.DataFrame(partial_corr, index=matrix.index, columns=matrix.columns)

# Save to tsv
partial_corr_df.to_csv('partial_correlation_matrix.tsv', sep='\t', index=True)